<a href="https://colab.research.google.com/github/triastrale/stemming/blob/snowball_stemmer/StemmingInggris_Kel7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk pandas

In [ ]:
import pandas as pd
import nltk
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Download dulu data nltk (kalau baru pertama kali pakai)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
# Baca file CSV
df = pd.read_csv('/content/Dataset_English - Sheet1.csv')


In [ ]:
# Bikin stemmer untuk English
stemmer = SnowballStemmer("english")

# Fungsi stemming untuk satu kalimat
def stemming_kalimat(kalimat):
    if pd.isnull(kalimat):
        return ""
    # Tokenisasi dan lowercase
    words = word_tokenize(kalimat.lower())
    # Buang tanda baca
    words = [word for word in words if word not in string.punctuation]
    # Buang stopwords
    words = [word for word in words if word not in stop_words]
    # Lakukan stemming
    stemmed_words = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed_words)


In [ ]:
# Terapkan stemming ke kolom 'judul' dan 'isi'
df['judul_stemmed'] = df['Judul'].apply(stemming_kalimat)
df['isi_stemmed'] = df['Isi'].apply(stemming_kalimat)

# Lihat hasilnya
df[['Judul', 'judul_stemmed', 'Isi', 'isi_stemmed']].head()


,Judul,judul_stemmed,Isi,isi_stemmed
0,7 Drugs That Changed the World,7 drug chang world,"People have swallowed elixirs, inhaled vapors,...",peopl swallow elixir inhal vapor appli ointmen...
1,A 'Black Snape' in the new Harry Potter seems ...,black snape new harri potter seem design caus ...,Ignore the fuss: Paapa Essiedu is a brilliant ...,ignor fuss paapa essiedu brilliant actor bring...
2,Alexander Fleming,alexand fleme,In 1928 Alexander Fleming (1881–1955) discover...,1928 alexand fleme 1881–1955 discov penicillin...
3,An anonymous 4chan post could help solve a 25-...,anonym 4chan post could help solv 25-year-old ...,A 4chan poster may have solved part of a very ...,4chan poster may solv part tricki math problem...
4,Are breakfast cereals really good for us?,breakfast cereal realli good us,Fortified breakfast cereals can be a useful so...,fortifi breakfast cereal use sourc fibr vitami...


In [ ]:
# Simpan hasil stemming ke CSV baru
df.to_csv('hasil_stemming2.csv', index=False, encoding='utf-8-sig')

print("Sukses stemming! Hasil disimpan ke hasil_stemming.csv")


Sukses stemming! Hasil disimpan ke hasil_stemming.csv


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Mean Word Count (MWC)
df['judul_word_count'] = df['judul_stemmed'].apply(lambda x: len(x.split()))
df['isi_word_count'] = df['isi_stemmed'].apply(lambda x: len(x.split()))
mean_word_count = (df['judul_word_count'].mean() + df['isi_word_count'].mean()) / 2

# Untuk Understemming & Overstemming (heuristik menggunakan TF-IDF cosine similarity)
# Kita bandingkan teks sebelum dan sesudah stemming
vectorizer = TfidfVectorizer()

def calc_similarity(before, after):
    tfidf = vectorizer.fit_transform([before, after])
    return cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]

# Hitung skor similarity untuk setiap baris antara sebelum dan sesudah stemming
df['judul_sim'] = df.apply(lambda row: calc_similarity(str(row['Judul']), row['judul_stemmed']), axis=1)
df['isi_sim'] = df.apply(lambda row: calc_similarity(str(row['Isi']), row['isi_stemmed']), axis=1)

# Simpulan heuristik (semakin rendah similarity, kemungkinan overstemming/understemming)
understemming_index = 1 - df['judul_sim'].mean()  # asumsinya: low similarity = understemming
overstemming_index = 1 - df['isi_sim'].mean()     # heuristic; perlu ground truth untuk validasi akurat

# Cetak hasil evaluasi
print(f"Mean Word Count (MWC): {mean_word_count:.2f}")
print(f"Understemming Index (judul): {understemming_index:.2f}")
print(f"Overstemming Index (isi): {overstemming_index:.2f}")


Mean Word Count (MWC): 325.43
Understemming Index (judul): 0.71
Overstemming Index (isi): 0.84
